### Set up

In [353]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
simple_train_data = pd.read_csv('data/simple-train.csv')
simple_test_data = pd.read_csv('data/simple-test.csv')
netflix_train_data = pd.read_csv('data/netflix-train.csv')
netflix_test_data = pd.read_csv('data/netflix-test.csv')
netflix_train_data_no_demo = pd.read_csv('data/netflix-train.csv', usecols=lambda x: x not in ['Demographic'])
netflix_test_data_no_demo = pd.read_csv('data/netflix-test.csv', usecols=lambda x: x not in ['Demographic'])
ancestry_train_data = pd.read_csv('data/ancestry-train.csv')
ancestry_test_data = pd.read_csv('data/ancestry-test.csv')
heart_train_data = pd.read_csv('data/heart-train.csv')
heart_test_data = pd.read_csv('data/heart-test.csv')
heart_train_data_no_demo = pd.read_csv('data/heart-train.csv', usecols=lambda x: x not in ['Demographic'])
heart_test_data_no_demo = pd.read_csv('data/heart-test.csv', usecols=lambda x: x not in ['Demographic'])
simple_train_data.head()

# Magic numbers
N = 1
MAX = 3000
ETA = 0.0001


### Review of Logistic Regression

Logistic Regression is a classification algorithm based on MLE, it assumes that: $$p = P(Y = 1 | X = x) = \sigma (z)$$, where $z = \theta^T x$.

Because Y is a random variable of Bernoulli distribution, therefore $$P(Y = y | X = x) = p^y (1 - p)^{1 - y}$$

Given the training data, we can compute the likelihood function 
$$L(\theta) = \prod_{i = 1}^{n}P(Y = y^{(i)} | X = x^{(i)}) = \prod_{i = 1}^{n}p^{y^{(i)}}(1 - p)^{1 - y^{(i)}}$$
$$LL(\theta) = \sum_{i = 1}^{n}y^{(i)} \log p + (1 - y^{(i)}) \log (1 - p)$$

To find the set of $\theta$ to maximize $LL(\theta)$, we use Gradient Descent

$$\frac{\partial LL(\theta)}{\partial \theta_j} = \sum_{i = 1}^{n} \left[y^{(i)} - \sigma (\theta ^T \mathbf{x^{(i)}}) \right]x_{j}^{(i)}$$
$$\theta_{j}^{\text{new}} = \theta_{j}^{\text{old}} + \eta \cdot \frac{\partial \, LL(\theta^{\text{old}})}{\partial \theta_j^{\text{old}}}$$

To let Gradient Descent, try different starting points to find the set of $\theta$ that maximizes $LL(\theta)$

### Implement Logistic Regression

In [337]:
def choose_theta(data):
    """
    Choose the value of thetas from range [-0.001, 0.001]

    Args:
        data (pd.DataFrame): The training data

    Returns:
        theta (pd.Series): The randomly chosen thetas
    """
    theta = pd.Series(np.random.uniform(-0.001, 0.001, data.shape[1] - 1))
    return theta
random_theta = choose_theta(netflix_train_data_no_demo)
random_theta

0     0.000059
1    -0.000127
2     0.000809
3     0.000560
4    -0.000702
5     0.000944
6     0.000328
7     0.000263
8    -0.000325
9    -0.000352
10    0.000931
11   -0.000725
12    0.000888
13    0.000385
14    0.000424
15    0.000026
16    0.000592
17   -0.000133
18   -0.000805
dtype: float64

In [339]:
def compute_likelihood(theta, data):
    """
    Compute the log likelihood for the given thetas 

    Args:
        theta (pd.Series): The specified set of thetas

    Returns:
        likelihood (Float): The computed log likelihood
    """
    epsilon = 1e-15  # avoid log(0)
    X = data.iloc[:, :-1].values  # features
    y = data['Label'].values      # labels

    z = X @ theta.values          # z = θᵀx
    p = 1 / (1 + np.exp(-z))      # sigmoid function

    log_likelihood = np.sum(y * np.log(p + epsilon) + (1 - y) * np.log(1 - p + epsilon))
    return log_likelihood
compute_likelihood(random_theta, netflix_train_data_no_demo)

np.float64(-3119.573131102357)

In [340]:
def gradient_descent(theta, data):
    """
    Operate Gradient Descent for one time, return the update theta and the corresponding likelihood

    Args:
        theta (pd.Series): The specified set of thetas
        data (pd.DataFrame): The training data

    Returns:
        theta (pd.Series):the updated thetas
        likelihood (Float): The corresponding likelihood
    """
    X = data.iloc[:, :-1].values  # features
    y = data['Label'].values      # labels
    m = len(y)

    z = X @ theta.values          # dot product: z = θ^T x
    p = 1 / (1 + np.exp(-z))      # sigmoid

    gradient = X.T @ (y - p)      # vectorized gradient
    new_theta = theta + ETA * gradient

    return pd.Series(new_theta, index=theta.index), compute_likelihood(new_theta, data)
gradient_descent(random_theta, netflix_train_data_no_demo)

(0     0.016152
 1     0.003773
 2    -0.000366
 3     0.000637
 4     0.020265
 5     0.004819
 6     0.003865
 7     0.017303
 8     0.006123
 9     0.002352
 10    0.007953
 11    0.004627
 12    0.009271
 13    0.003315
 14   -0.000073
 15    0.000789
 16    0.006642
 17    0.021376
 18    0.054187
 dtype: float64,
 np.float64(-3080.5810623969696))

In [341]:
def compute_theta(data):
    """
    Find the set of theta with the maximum likelihood for a single starting point of thetas

    Args:
        data (pd.DataFrame): The training data

    Returns:
        likelihood (Float): The maximum likelihood
        theta (pd.Series): The set of thetas with the maximum likelihood
    """
    # Initialize the starting point of thetas
    theta = choose_theta(data)
    # Keep doing Gradient Descent until converge
    for _ in range(MAX):
        # Use Gradient Descent to compute the new theta
        theta, likelihood = gradient_descent(theta, data)
    return likelihood, theta
compute_theta(netflix_train_data_no_demo)

(np.float64(-2628.928747732837),
 0     0.120248
 1    -0.106377
 2    -0.239292
 3    -0.196455
 4     0.155034
 5    -0.069361
 6    -0.077374
 7     0.112592
 8    -0.266414
 9    -0.198841
 10   -0.005488
 11   -0.036669
 12   -0.109316
 13   -0.152822
 14   -0.127012
 15   -0.100774
 16   -0.146622
 17    0.128349
 18    1.804069
 dtype: float64)

In [343]:
def optimize_theta(data):
    """
    Do Gradient Descent multiple times with different starting point 
    to find the set of thetas with the maximum likelihood

    Args:
        data (pd.DataFrame): The training data
    
    Returns:
        max_likelihood (Float): The maximum likelihood
        theta (pd.Series): The set of thetas with the maximum likelihood
    """
    # Initialize the best theta and the maximum likelihood
    best_theta = pd.Series()
    max_likelihood = float('-inf')
    # Do Gragient Descent multiple times with different starting points of thetas
    for _ in range(N):
        likelihood, theta = compute_theta(data)
        if likelihood > max_likelihood:
            best_theta = theta
            max_likelihood = likelihood
    return best_theta, max_likelihood
optimize_theta(netflix_train_data_no_demo)

(0     0.120248
 1    -0.106377
 2    -0.239292
 3    -0.196455
 4     0.155034
 5    -0.069361
 6    -0.077374
 7     0.112592
 8    -0.266414
 9    -0.198841
 10   -0.005488
 11   -0.036669
 12   -0.109316
 13   -0.152822
 14   -0.127012
 15   -0.100774
 16   -0.146622
 17    0.128349
 18    1.804069
 dtype: float64,
 np.float64(-2628.9287477328367))

In [344]:
def check_accuracy(test_data, train_data):
    """
    Use the trained model to predict the target value of the test data and check the accuracy

    Args:
        train_data (pd.DataFrame): The training data
        test_data (pd.DataFrame): The test data
    """
   # Compute the set of thetas trained from the training data
    theta, _ = optimize_theta(train_data)
    
    # Extract features and true labels from test data
    X_test = test_data.iloc[:, :-1].values  # shape: (m, n)
    y_true = test_data['Label'].values      # shape: (m,)

    # Compute predicted probabilities
    z = X_test @ theta.values               # θᵀx
    p = 1 / (1 + np.exp(-z))                # sigmoid function

    # Predict labels based on 0.5 threshold
    y_pred = (p >= 0.5).astype(int)

    # Compute accuracy
    accuracy = np.mean(y_pred == y_true)
    return accuracy
check_accuracy(simple_test_data, simple_train_data)

np.float64(0.75)

### a. Logistic Regression for simple data

In [345]:
N, MAX, ETA = 1, 10_000, 0.0001
check_accuracy(simple_test_data, simple_train_data)

np.float64(0.75)

### b. Logistic Regression for Netflix data

##### 1. Report the classification accuracy

In [346]:
N, MAX, ETA = 5, 3000, 0.0001
check_accuracy(netflix_test_data_no_demo, netflix_train_data_no_demo)

np.float64(0.68)

##### 2. Find the 5 strongest pridictors

In [347]:
# Compute the theta trained from the training data
likelihood, theta = compute_theta(netflix_train_data_no_demo)
# Find the indices of the 5 stonest pridictors
indices = theta.sort_values(ascending=False).head()
# Print the movie name of the 5 strongest pridictors
netflix_train_data_no_demo.columns[indices.index]

Index(['When Harry Met Sally', 'How to Lose a Guy in 10 Days',
       'What Women Want', '3 Idiots', 'La Vita E Bella'],
      dtype='object')

##### 3. The log likelihood when all the parameters are 0

$
LL(\theta) = \sum_{i = 1}^{n} \log \frac{1}{1 + e^0} = -n \log2
$

##### 4. The log likelihood after training

In [348]:
# Compute the log likelihood after training
likelihood, theta = compute_theta(netflix_train_data_no_demo)
likelihood

np.float64(-2628.9287477328367)

### c. Logistic Regression for acestry data

In [349]:
N, MAX, ETA = 1, 10_000, 0.0001
check_accuracy(ancestry_test_data, ancestry_train_data)

np.float64(0.8369565217391305)

### d. Logistic Regression for heart data

In [357]:
N, MAX, ETA = 1, 10_000, 0.0005
check_accuracy(heart_test_data_no_demo, heart_train_data_no_demo)

np.float64(0.7379679144385026)

In [358]:
N, MAX, ETA = 1, 10_000, 0.00002
check_accuracy(heart_test_data_no_demo, heart_train_data_no_demo)

np.float64(0.8770053475935828)

Explain the reason why smaller eta results higher accuracy:

A smaller learning rate $\eta$ allows Gradient Descent to make smaller, more precise updates, which helps the algorithm to converge more smoothly and reduces the chance of overshooting, allowing the computed $\theta$ to move closer to the optimal values

### e. The code copy (see the code above)